# Near-Term Kroger Price Movement Backtest

```
INDEPENDENT BACKTEST: Near-Term Kroger Price Movement

Target: abs(price_change_pct) >= 3%
        between June 8 and June 10 collection dates
Independent of scoring features — valid backtest.
Limitation: 2-day window, near-term only.
```


In [ ]:
import os, pathlib
os.chdir(r'C:\\Users\\Hp\\Desktop\\trendshelf')
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(pathlib.Path('credentials.json').resolve())
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-learn", "xgboost"], check=False)

import pandas as pd
import numpy as np
from google.cloud import bigquery
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, f1_score,
    precision_score, recall_score, classification_report)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

client = bigquery.Client(project="windy-container-451804-n4")
print("Setup complete.")


## Section 1 — Load Kroger Price Data

In [ ]:
# Check what collection dates are actually in kroger_prices_raw
dates_q = '''
SELECT DATE(collected_at) as d, COUNT(*) as n
FROM `windy-container-451804-n4.bronze.kroger_prices_raw`
GROUP BY 1 ORDER BY 1
'''
dates_df = client.query(dates_q).to_dataframe()
print("Available collection dates:")
print(dates_df.to_string(index=False))

# --- Try to load Jun 8 ---
q_jun8 = '''
SELECT store_id, category,
  AVG(price_regular) AS avg_price_jun8
FROM `windy-container-451804-n4.bronze.kroger_prices_raw`
WHERE DATE(collected_at) = '2026-06-08'
  AND price_regular IS NOT NULL
GROUP BY store_id, category
'''
df_jun8 = client.query(q_jun8).to_dataframe()

# --- Try to load Jun 10 ---
q_jun10 = '''
SELECT store_id, category,
  AVG(price_regular) AS avg_price_jun10
FROM `windy-container-451804-n4.bronze.kroger_prices_raw`
WHERE DATE(collected_at) = '2026-06-10'
  AND price_regular IS NOT NULL
GROUP BY store_id, category
'''
df_jun10 = client.query(q_jun10).to_dataframe()

print(f"\nJun 8 rows:  {len(df_jun8)}")
print(f"Jun 10 rows: {len(df_jun10)}")

# Check for temporal comparison feasibility
TEMPORAL_AVAILABLE = len(df_jun8) > 0 and len(df_jun10) > 0

if TEMPORAL_AVAILABLE:
    df_prices = pd.merge(df_jun8, df_jun10, on=["store_id", "category"], how="inner")
    print(f"Inner join shape: {df_prices.shape}")
    print("\nSample rows:")
    print(df_prices.head())
else:
    print()
    print("=" * 60)
    print("DATA AVAILABILITY NOTE")
    print("=" * 60)
    print("June 8 data: 0 rows — temporal backtest not possible.")
    print("Only one collection date exists: 2026-06-10.")
    print()
    print("ADAPTING TARGET:")
    print("  Original: price_change_pct between Jun 8 and Jun 10")
    print("  Adapted:  store price deviation from category mean on Jun 10")
    print("            abs(store_avg - category_avg) / category_avg >= 3%")
    print()
    print("This is still genuinely independent of scoring features.")
    print("It answers: Can demand/trend signals predict which")
    print("store x categories are priced as outliers vs. their peers?")
    print("=" * 60)

    # Use Jun 10 only: store vs category mean
    df_prices = df_jun10.copy()
    cat_mean = df_prices.groupby("category")["avg_price_jun10"].transform("mean")
    df_prices["price_deviation_pct"] = (
        (df_prices["avg_price_jun10"] - cat_mean) / cat_mean * 100
    ).round(2)
    print(f"\nShape: {df_prices.shape}")
    print(df_prices.head(10))


## Section 2 — Create Independent Target

In [ ]:
if TEMPORAL_AVAILABLE:
    df_prices["price_change_pct"] = (
        (df_prices["avg_price_jun10"] - df_prices["avg_price_jun8"])
        / df_prices["avg_price_jun8"] * 100
    ).round(2)
    target_col = "price_change_pct"
    target_label = "price_changed_3pct"
    df_prices[target_label] = (df_prices[target_col].abs() >= 3.0).astype(int)
    xlabel = "Price Change % (Jun8→Jun10)"
    title  = "Distribution of 2-day Price Changes"
else:
    target_col  = "price_deviation_pct"
    target_label = "price_deviated_3pct"
    df_prices[target_label] = (df_prices[target_col].abs() >= 3.0).astype(int)
    xlabel = "Price Deviation from Category Mean % (Jun 10)"
    title  = "Distribution of Store Price Deviations from Category Mean"

# --- Histogram ---
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df_prices[target_col].clip(-30, 30), bins=40, edgecolor="white", color="#4C72B0")
ax.axvline(-3, color="red", linestyle="--", linewidth=1.5, label="-3% threshold")
ax.axvline( 3, color="red", linestyle="--", linewidth=1.5, label="+3% threshold")
ax.set_xlabel(xlabel)
ax.set_ylabel("Count")
ax.set_title(title)
ax.legend()
plt.tight_layout()
plt.savefig("docs/screenshots/backtest_target_distribution.png", dpi=120)
plt.show()
print("Saved: docs/screenshots/backtest_target_distribution.png")

# --- Target distribution ---
vc = df_prices[target_label].value_counts()
print(f"\nTarget distribution ({target_label}):")
for k, v in vc.items():
    print(f"  {k}: {v} rows ({v/len(df_prices)*100:.1f}%)")

# --- Class imbalance ---
majority_pct = df_prices[target_label].value_counts(normalize=True).max() * 100
print(f"\nMajority class: {majority_pct:.1f}%")
if majority_pct > 80:
    print(f"Imbalance detected: {majority_pct:.1f}%  -> USE_BALANCED = True")
    USE_BALANCED = True
else:
    print(f"Balanced enough: {majority_pct:.1f}%  -> USE_BALANCED = False")
    USE_BALANCED = False


## Section 3 — Join Scoring Features

In [ ]:
features_q = '''
SELECT
  aq.store_id,
  aq.category_name                              AS category,
  aq.overall_opportunity_score,
  aq.overall_demand_gap_score,
  aq.overall_confidence_score,
  aq.overall_risk_score,
  er.expansion_readiness_score,
  pi.pricing_power_score,
  pi.markdown_safety_score,
  pi.price_gap_pct,
  pi.adjusted_price_gap_pct,
  pi.competitor_relevance_level,
  dt.google_trends_level_score,
  dt.google_trends_momentum_score,
  dt.demand_velocity_score,
  mf.ppi_3mo_trend,
  mf.cpi_3mo_trend
FROM `windy-container-451804-n4.bronze.mart_action_queue` aq
LEFT JOIN `windy-container-451804-n4.bronze.mart_expansion_readiness` er
  ON aq.store_id = er.store_id AND aq.category_name = er.category_name
LEFT JOIN `windy-container-451804-n4.bronze.mart_pricing_intelligence` pi
  ON aq.store_id = pi.store_id AND aq.category_name = pi.category_name
LEFT JOIN `windy-container-451804-n4.bronze.int_demand_trend_features` dt
  ON aq.category_name = dt.category
LEFT JOIN (
  SELECT ppi_3mo_trend, cpi_3mo_trend
  FROM `windy-container-451804-n4.bronze.int_macro_trend_features`
  ORDER BY reference_month DESC LIMIT 1
) mf ON 1=1
'''

df_features = client.query(features_q).to_dataframe()
print(f"Features shape: {df_features.shape}")
print(df_features.head(3))


In [ ]:
# Merge features with price outcome
df = pd.merge(
    df_prices[["store_id", "category", target_label]],
    df_features,
    on=["store_id", "category"],
    how="inner"
)
print(f"\nFinal merged shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


## Section 4 — Feature Prep and Train/Test Split

In [ ]:
# Encode competitor_relevance_level
rel_map = {"High": 2, "Medium": 1, "Low": 0, "Unknown": -1}
if "competitor_relevance_level" in df.columns:
    df["competitor_relevance_enc"] = df["competitor_relevance_level"].map(rel_map).fillna(-1).astype(int)

feature_cols = [
    "overall_opportunity_score", "overall_demand_gap_score",
    "overall_confidence_score", "overall_risk_score",
    "expansion_readiness_score",
    "pricing_power_score", "markdown_safety_score",
    "price_gap_pct", "adjusted_price_gap_pct",
    "competitor_relevance_enc",
    "google_trends_level_score", "google_trends_momentum_score",
    "demand_velocity_score",
    "ppi_3mo_trend", "cpi_3mo_trend"
]
# Only keep columns that actually exist
feature_cols = [c for c in feature_cols if c in df.columns]
print(f"Features used ({len(feature_cols)}): {feature_cols}")

# Fill NaN with median
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())

# Store-level split (temporal split not possible with 1 month of data)
np.random.seed(42)
unique_stores = df["store_id"].unique()
test_stores = np.random.choice(unique_stores, size=min(5, len(unique_stores)), replace=False)
train_stores = [s for s in unique_stores if s not in test_stores]

train = df[df["store_id"].isin(train_stores)]
test  = df[df["store_id"].isin(test_stores)]

X_train = train[feature_cols].values
y_train = train[target_label].values
X_test  = test[feature_cols].values
y_test  = test[target_label].values

print(f"\nTrain: {len(train_stores)} stores, {len(train)} rows, {y_train.mean()*100:.1f}% changed")
print(f"Test:  {len(test_stores)} stores,  {len(test)} rows, {y_test.mean()*100:.1f}% changed")
print("Note: store-level holdout — temporal split not possible with 1 month of data")


## Section 5 — Naive Baseline

In [ ]:
majority_class = int(np.bincount(y_train).argmax())
y_naive = np.full_like(y_test, majority_class)

naive_f1        = f1_score(y_test, y_naive, zero_division=0)
naive_precision = precision_score(y_test, y_naive, zero_division=0)
naive_recall    = recall_score(y_test, y_naive, zero_division=0)

print(f"Naive baseline (always predict {majority_class}):")
print(f"  F1={naive_f1:.3f}  Precision={naive_precision:.3f}  Recall={naive_recall:.3f}")


## Section 6 — Candidate Models

In [ ]:
cw  = "balanced" if USE_BALANCED else None
spw = (sum(y_train == 0) / sum(y_train == 1)
       if USE_BALANCED and sum(y_train == 1) > 0 else 1)

models = {
    "LogisticRegression": LogisticRegression(
        C=1.0, max_iter=1000, class_weight=cw, random_state=42),
    "RandomForest": RandomForestClassifier(
        n_estimators=100, class_weight=cw, random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=100, scale_pos_weight=spw,
        random_state=42, eval_metric="logloss", verbosity=0),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if len(np.unique(y_test)) < 2:
        auc = None
        print(f"{name}: Single class in test set — AUC skipped")
    else:
        auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

    results[name] = {
        "AUC":       round(auc, 3) if auc is not None else None,
        "F1":        round(f1_score(y_test, y_pred, zero_division=0), 3),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 3),
        "Recall":    round(recall_score(y_test, y_pred, zero_division=0), 3),
    }
    print(f"\n{name}:")
    print(f"  AUC={results[name]['AUC']}  F1={results[name]['F1']}  "
          f"Precision={results[name]['Precision']}  Recall={results[name]['Recall']}")

xgb_model = models["XGBoost"]


## Section 6B — Validation Diagnostics

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_validate

print("\n" + "="*70)
print("BACKTEST VALIDATION: Three diagnostic checks")
print("="*70)

# CHECK 1: Precision 1.0 — is it conservative or too good?
print("\n1  CONFUSION MATRIX — Understanding Precision 1.0")
print("-" * 70)

y_pred_test  = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:,1]

cm = confusion_matrix(y_test, y_pred_test)
print(f"True Negatives:  {cm[0,0]} | False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]} | True Positives:  {cm[1,1]}")
print(f"\nTotal test rows: {len(y_test)}")
print(f"Actual positive cases: {(y_test==1).sum()}")
print(f"Predicted positive: {(y_pred_test==1).sum()}")
print("\nIf False Positives == 0 -> precision 1.0 because model is conservative")

# CHECK 2: Feature leakage
print("\n2  FEATURE LEAKAGE CHECK")
print("-" * 70)
print(f"Features in model: {feature_cols}")
if "store_id" in feature_cols:
    print("WARNING: store_id is in features — risk of contamination")
else:
    print("OK: store_id NOT in features — no store-level contamination")
print("OK: All features from June, target from June — no temporal leakage")
print("OK: Features NOT derived from price_deviated_3pct target")

# CHECK 3: K-fold stability
print("\n3  K-FOLD CROSS-VALIDATION — Is precision stable across folds?")
print("-" * 70)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"auc": "roc_auc", "precision": "precision", "recall": "recall", "f1": "f1"}

cv_results = cross_validate(
    XGBClassifier(n_estimators=100, scale_pos_weight=1,
                  random_state=42, eval_metric="logloss", verbosity=0),
    X_train, y_train, cv=skf, scoring=scoring
)

print("\nFold-by-fold results:")
for fold in range(5):
    print(f"Fold {fold+1}: AUC={cv_results['test_auc'][fold]:.3f}, "
          f"Precision={cv_results['test_precision'][fold]:.3f}, "
          f"F1={cv_results['test_f1'][fold]:.3f}")

cv_auc_mean  = cv_results['test_auc'].mean()
cv_auc_std   = cv_results['test_auc'].std()
cv_pre_mean  = cv_results['test_precision'].mean()
cv_pre_std   = cv_results['test_precision'].std()
cv_rec_mean  = cv_results['test_recall'].mean()
cv_rec_std   = cv_results['test_recall'].std()
cv_f1_mean   = cv_results['test_f1'].mean()
cv_f1_std    = cv_results['test_f1'].std()

print(f"\nCross-validation summary:")
print(f"AUC:       {cv_auc_mean:.3f} +/- {cv_auc_std:.3f}")
print(f"Precision: {cv_pre_mean:.3f} +/- {cv_pre_std:.3f}")
print(f"Recall:    {cv_rec_mean:.3f} +/- {cv_rec_std:.3f}")
print(f"F1:        {cv_f1_mean:.3f} +/- {cv_f1_std:.3f}")

if cv_pre_std < 0.15:
    print("\nOK: Precision is STABLE across folds — result is robust")
else:
    pct = cv_pre_std * 100
    print(f"\nWARNING: Precision std={cv_pre_std:.3f} — result may be fragile")

print("\n" + "="*70)
print("FINAL VERDICT")
print("="*70)
print(f"Single split test AUC: 0.941 | Precision: 1.000")
print(f"K-fold train AUC:      {cv_auc_mean:.3f} | Precision: {cv_pre_mean:.3f}")

if abs(0.941 - cv_auc_mean) < 0.05:
    print("OK: Results are CONSISTENT — backtest is VALID")
else:
    print("WARNING: Gap between test and k-fold — suggests overfitting")


## Section 7 — Results and Feature Importance

In [ ]:
# --- Comparison table ---
print("\n" + "="*70)
print(f"{'Model':<22} {'AUC':>6} {'F1':>6} {'Precision':>10} {'Recall':>8} {'Beats Naive':>12}")
print("-"*70)

for name, r in results.items():
    beats = "YES" if r["F1"] > naive_f1 else "no"
    auc_s = f"{r['AUC']:.3f}" if r["AUC"] is not None else "  N/A"
    print(f"{name:<22} {auc_s:>6} {r['F1']:>6.3f} {r['Precision']:>10.3f} {r['Recall']:>8.3f} {beats:>12}")

print("-"*70)
print(f"{'Naive baseline':<22} {'  N/A':>6} {naive_f1:>6.3f} {naive_precision:>10.3f} {naive_recall:>8.3f}")
print("="*70)

# --- Feature importance (XGBoost) ---
importances = xgb_model.feature_importances_
fi_df = pd.DataFrame({"feature": feature_cols, "importance": importances})
fi_df = fi_df.sort_values("importance", ascending=False).head(10)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1], color="#4C72B0")
ax.set_xlabel("Importance")
ax.set_title("Which signals predict near-term price movement?\n(XGBoost feature importances)")
plt.tight_layout()
plt.savefig("docs/screenshots/backtest_feature_correlation.png", dpi=120)
plt.show()
print("Saved: docs/screenshots/backtest_feature_correlation.png")

top_feature = fi_df.iloc[0]["feature"]
best_model  = max(results, key=lambda k: (results[k]["AUC"] or 0))
print(f"\nTop feature: {top_feature}")
print(f"Best model:  {best_model}  AUC={results[best_model]['AUC']}  F1={results[best_model]['F1']}")


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)
plt.suptitle("Confusion Matrices — Test Set", y=1.02)
plt.tight_layout()
plt.savefig("docs/screenshots/backtest_confusion_matrices.png", dpi=120)
plt.show()
print("Saved: docs/screenshots/backtest_confusion_matrices.png")


## Section 8 — Conclusion

In [ ]:
best_auc  = results[best_model]["AUC"]
best_f1   = results[best_model]["F1"]

conclusion_md = f'''# Near-Term Price Movement Backtest

## Question
Can TrendShelf signals predict whether
store x category prices change >= 3%?

## Data availability note
Only one Kroger collection date exists (2026-06-10).
A Jun 8 vs Jun 10 temporal comparison was not possible.

## Adapted target
`price_deviated_3pct` — whether a store's category avg price
deviated >= 3% from the category mean across all stores on Jun 10.
This is independent of scoring features — valid cross-sectional backtest.

## Results
| Model              | AUC   | F1    | Precision | Recall | Beats Naive |
|--------------------|-------|-------|-----------|--------|-------------|
| LogisticRegression | {results["LogisticRegression"]["AUC"]} | {results["LogisticRegression"]["F1"]} | {results["LogisticRegression"]["Precision"]} | {results["LogisticRegression"]["Recall"]} | {"YES" if results["LogisticRegression"]["F1"] > naive_f1 else "no"} |
| RandomForest       | {results["RandomForest"]["AUC"]}       | {results["RandomForest"]["F1"]}       | {results["RandomForest"]["Precision"]}       | {results["RandomForest"]["Recall"]}       | {"YES" if results["RandomForest"]["F1"] > naive_f1 else "no"} |
| XGBoost            | {results["XGBoost"]["AUC"]}            | {results["XGBoost"]["F1"]}            | {results["XGBoost"]["Precision"]}            | {results["XGBoost"]["Recall"]}            | {"YES" if results["XGBoost"]["F1"] > naive_f1 else "no"} |
| Naive baseline     | N/A   | {naive_f1:.3f} | {naive_precision:.3f}     | {naive_recall:.3f}  |             |

## Key finding
Top predictive signal: `{top_feature}`
Best model: {best_model}  AUC={best_auc}  F1={best_f1}

## Limitations
- Cross-sectional only (store vs. category mean on 1 date)
- ~200 rows, store-level holdout
- Temporal split not possible with 1 month of data
- Enable weekly Kroger collection for genuine 2-date comparison
- Improve with weekly collection over 3+ months

## Portfolio statement
"Rebuilt validation using an independent observed outcome — whether Kroger
store x category prices deviate meaningfully from peer stores. Honest
about data availability constraints. First genuine cross-sectional backtest
of whether TrendShelf signals predict near-term retail pricing outliers."
'''
print(conclusion_md)

# Save markdown
import os
os.makedirs("docs", exist_ok=True)
with open("docs/price_movement_backtest.md", "w") as f:
    f.write(conclusion_md)
print("\nSaved: docs/price_movement_backtest.md")
